In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram

In [ ]:
path = 'HelmLiteHeatmap'
#alternatives:
    #'HelmLiteHeatmap_0.9_0.1'
    #'HelmLiteHeatmap_0.1_0.4'

df = pd.read_pickle(f'../results/{path}/heatmap_data_compressed.pkl')

In [ ]:
M = df.pivot(index='source', columns='target', values='auroc')
M = M.fillna(M.median(axis=1), axis=0)

D_rows = pdist(M.values, lambda u, v: np.mean(np.abs(u - v)))
Z_rows = linkage(D_rows, method='ward')

dendro = dendrogram(
    Z_rows,
    labels=M.index.tolist(),
    orientation='right',
    leaf_font_size=8,
    color_threshold=0.1
)

order = dendro['leaves']
ordered_sources = [M.index[i] for i in order]
#ordered_sources.reverse() # used in 0.9_0.1 setting

M_reordered = M.loc[ordered_sources, ordered_sources]

cmap = sns.color_palette("magma", as_cmap=True)

fig, ax = plt.subplots(figsize=(13.636, 12))
im = ax.imshow(
    M_reordered.values,
    aspect="auto",
    cmap=cmap,
    interpolation="nearest",                 
    vmin=M_reordered.to_numpy().min(),   
    vmax=M_reordered.to_numpy().max()
)
cbar = fig.colorbar(im, ax=ax, label="AUROC")

ax.set_xticks(range(len(ordered_sources)))
ax.set_xticklabels(ordered_sources, rotation=90, fontsize=6)
ax.set_yticks(range(len(M_reordered.index)))
ax.set_yticklabels(M_reordered.index, fontsize=6)

if path == 'HelmLiteHeatmap':
    CB = {
        "blue":   "#648FFF",
        "orange": "#FE6100",
        "red":    "#DC267F",
        "purple": "#785EF0",
        "yellow": "#FFB000",
    }

    x_group_a = {'meta_llama-3.1-8b-instruct-turbo', 'mistralai_open-mistral-nemo-2407', 'google_gemini-1.5-pro-preview-0409'}
    x_group_b = {'microsoft_phi-2', 'cohere_command-light'}

    y_group_a = {'mistralai_mixtral-8x22b', 'deepseek-ai_deepseek-llm-67b-chat', 'qwen_qwen1.5-72b'}
    y_group_b = {'cohere_command-light', 'microsoft_phi-2', 'AlephAlpha_luminous-extended'}
    y_group_c = {'mistralai_open-mistral-nemo-2407', 'mistralai_mistral-large-2402', 'mistralai_mistral-small-2402'}

    x_color = {m: CB["orange"] for m in x_group_a}
    x_color.update({m: CB["purple"] for m in x_group_b})

    y_color = {m: CB["blue"] for m in y_group_a}
    y_color.update({m: CB["red"] for m in y_group_b})
    y_color.update({m: CB["yellow"] for m in y_group_c})

    for t in ax.get_xticklabels():
        lab = t.get_text()
        if lab in x_color:
            t.set_color(x_color[lab])
            t.set_fontweight("bold")

    for t in ax.get_yticklabels():
        lab = t.get_text()
        if lab in y_color:
            t.set_color(y_color[lab])
            t.set_fontweight("bold")

    for t in ax.get_xticklabels(): 
        if t.get_text() not in x_color: t.set_color("#666666")
    for t in ax.get_yticklabels(): 
        if t.get_text() not in y_color: t.set_color("#666666")

plt.savefig(f"../results/figures/Barrier2_{path}.pdf", dpi=300, bbox_inches='tight')
fig.tight_layout()
plt.show()